# 02 - Prepare the IDRiD data

This notebook takes the findings from EDA and prepares labels and split lists for training. It does not edit the original images. Run `01_IDRiD_EDA.ipynb` first so the hash and exclusion files exist.

## Cell 1 - Load tools and find the project

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from torchvision import transforms

In [ ]:
ROOT_OVERRIDE = None  # Set a folder path here only if automatic search fails.
if ROOT_OVERRIDE:
    ROOT = Path(ROOT_OVERRIDE).expanduser().resolve()
else:
    here = Path.cwd().resolve()
    choices = [here, *here.parents, Path.home() / "Desktop" / "retinal-dr-classification"]
    ROOT = next((p for p in choices if (p / "data" / "IDRiD").is_dir()), None)
    if ROOT is None:
        raise FileNotFoundError("Set ROOT_OVERRIDE to the project folder.")
RUN_DIR = ROOT / "reports" / "IDRiD_clean_run"
images = pd.read_csv(RUN_DIR / "dataset_manifest.csv")
hashes = pd.read_csv(RUN_DIR / "exact_image_hashes.csv")
excluded = pd.read_csv(RUN_DIR / "exact_cross_split_duplicates_excluded.csv")
print("Images before exclusions:", len(images))
print("Exact cross-split rows to exclude:", len(excluded))

## Cell 2 - Make the two-class label

We keep the original grade for reference. `label` is what the model will learn: 0 = Normal, 1 = DR.

In [ ]:
images["label"] = (images["grade"] >= 1).astype(int)
images["class_name"] = images["label"].map({0: "Normal", 1: "DR"})
display(images[["image_id", "split", "grade", "label", "class_name"]].head())

## Cell 3 - Exclude only the unresolved exact-copy pair

We remove the two conflicting duplicate records from the modeling manifest. We do not delete their original image files.

In [ ]:
excluded_keys = set(zip(excluded["split"], excluded["image_id"]))
images["is_excluded"] = [
    (row.split, row.image_id) in excluded_keys for row in images.itertuples()
]
clean_images = images[~images["is_excluded"]].copy().reset_index(drop=True)
print("Kept for modeling:", len(clean_images))
print("Excluded:", int(images["is_excluded"].sum()))
display(clean_images.groupby(["split", "class_name"]).size().unstack(fill_value=0))

## Cell 4 - Add exact-image groups

Exact copies inside the official training set should not be split between training and validation. We attach the SHA-256 fingerprint found by the EDA notebook, then keep each fingerprint group together.

In [ ]:
clean_images = clean_images.merge(
    hashes[["split", "image_id", "sha256"]], on=["split", "image_id"], how="left"
)
print("Images without a hash:", int(clean_images["sha256"].isna().sum()))
print("Exact-copy groups still present:", int((clean_images.groupby("sha256").size() > 1).sum()))

## Cell 5 - Make train, validation, and test lists

The official test split stays separate. We divide the remaining official training images into five folds by grade class and exact-image group, then use the first fold as validation. This gives about 80% for training and 20% for validation while keeping exact copies together.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

train_source = clean_images[clean_images["split"] == "train"].reset_index(drop=True)
test_data = clean_images[clean_images["split"] == "test"].reset_index(drop=True)
folds = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=20261008)
train_rows, validation_rows = next(folds.split(
    X=np.zeros(len(train_source)),
    y=train_source["label"],
    groups=train_source["sha256"]
))
train_data = train_source.iloc[train_rows].copy().reset_index(drop=True)
validation_data = train_source.iloc[validation_rows].copy().reset_index(drop=True)

print("Training images:", len(train_data))
print("Validation images:", len(validation_data))
print("Eligible test images:", len(test_data))

## Cell 6 - Check class counts and duplicate separation

This is a check before training. We expect 329 training, 83 validation, and 102 test images. No exact SHA group should appear in two partitions.

In [ ]:
parts = {"train": train_data, "validation": validation_data, "test": test_data}
counts = pd.DataFrame({
    name: part["label"].value_counts().reindex([0, 1], fill_value=0)
    for name, part in parts.items()
}).rename(index={0: "Normal", 1: "DR"})
display(counts)

assert set(train_data.sha256).isdisjoint(set(validation_data.sha256))
assert set(train_data.sha256).isdisjoint(set(test_data.sha256))
assert set(validation_data.sha256).isdisjoint(set(test_data.sha256))
print("Check passed: no exact image group crosses partitions.")

## Cell 7 - Save the split lists

These CSV files make the split repeatable and easy to inspect.

In [ ]:
train_data.to_csv(RUN_DIR / "train_manifest.csv", index=False)
validation_data.to_csv(RUN_DIR / "validation_manifest.csv", index=False)
test_data.to_csv(RUN_DIR / "official_test_manifest.csv", index=False)
print("Saved split lists in:", RUN_DIR)

## Cell 8 - Define image preprocessing

The original photos are 4288 x 2848. We resize/crop them **when they are loaded into the model**, not by overwriting the source files. All model inputs become 224 x 224. A random horizontal flip is used only during training. Validation and test images are not randomly augmented.

In [ ]:
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

evaluation_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])
print("Preprocessing is ready. Original image files are unchanged.")

## Cell 9 - Preview one resized image

This preview shows the model input size. Normalization is numerical and is not intended to make the original image look different on disk.

In [ ]:
example_path = ROOT / train_data.loc[0, "relative_path"]
with Image.open(example_path) as photo:
    print("Original size:", photo.size)
    preview = evaluation_transform(photo.convert("RGB"))
print("Model input size:", tuple(preview.shape))
plt.imshow(preview.permute(1, 2, 0).numpy() * std + mean)
plt.axis("off")
plt.show()

### Preprocessing decisions in plain words

- The two exact copies with conflicting grades were excluded from the experiment, but the original files were kept.
- Five remaining exact-copy groups are kept together by the split.
- Near-duplicate candidates are not deleted without visual confirmation.
- Imbalance is handled later with weighted loss; we do not oversample or remove images.
- Images are resized/cropped and augmented in memory during training; the originals are not changed.